In [ ]:
!pip install plotly pandas requests nbformat -q

import requests
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timedelta

API_URL = 'https://djtnqbvkhqftmtnsityx.supabase.co/rest/v1/'
API_KEY = 'sb_publishable_NnOjc1iJWmA7j418h79mEg_AHh9h49u'
HEADERS = {'apikey': API_KEY, 'Authorization': f'Bearer {API_KEY}'}

def query_supabase(endpoint, select='*', filters=None, limit=10000):
    """Query Supabase REST API with filters."""
    params = {}
    if select: params['select'] = select
    if filters:
        for k, v in filters.items():
            params[k] = f'eq.{v}'
    if limit: params['limit'] = limit
    r = requests.get(f'{API_URL}{endpoint}', headers=HEADERS, params=params)
    if r.status_code == 200:
        return pd.DataFrame(r.json())
    else:
        print(f'Error {r.status_code}: {r.text[:200]}')
        return pd.DataFrame()

In [ ]:
# Query daily funding for equity perps
equity_df = query_supabase(
    'daily_funding',
    select='venue,symbol,date,avg_rate_bps,asset_class,is_weekend',
    filters={'asset_class': 'equity'},
    limit=50000
)
equity_df['date'] = pd.to_datetime(equity_df['date'])
equity_df['avg_rate_bps'] = pd.to_numeric(equity_df['avg_rate_bps'], errors='coerce')
equity_df = equity_df.dropna(subset=['avg_rate_bps'])
equity_df['is_weekend'] = equity_df['is_weekend'].astype(str).str.lower().isin(['true', '1', 't', 'yes'])

print(f"Equity perps: {len(equity_df):,} rows, symbols: {equity_df['symbol'].unique().tolist()}")
equity_df.head()

In [ ]:
# Query daily funding for crypto perps (control group)
crypto_df = query_supabase(
    'daily_funding',
    select='venue,symbol,date,avg_rate_bps,asset_class,is_weekend',
    filters={'asset_class': 'crypto'},
    limit=50000
)
crypto_df['date'] = pd.to_datetime(crypto_df['date'])
crypto_df['avg_rate_bps'] = pd.to_numeric(crypto_df['avg_rate_bps'], errors='coerce')
crypto_df = crypto_df.dropna(subset=['avg_rate_bps'])
crypto_df['is_weekend'] = crypto_df['is_weekend'].astype(str).str.lower().isin(['true', '1', 't', 'yes'])

print(f"Crypto perps: {len(crypto_df):,} rows")
crypto_df.head()

In [ ]:
# Bar chart: weekend vs weekday avg funding by asset class
equity_df['day_type'] = equity_df['is_weekend'].map({True: 'Weekend', False: 'Weekday'})
crypto_df['day_type'] = crypto_df['is_weekend'].map({True: 'Weekend', False: 'Weekday'})

equity_summary = equity_df.groupby('day_type')['avg_rate_bps'].mean().reset_index()
equity_summary['asset_class'] = 'Equity'
crypto_summary = crypto_df.groupby('day_type')['avg_rate_bps'].mean().reset_index()
crypto_summary['asset_class'] = 'Crypto'

combined = pd.concat([equity_summary, crypto_summary])

fig = px.bar(
    combined, x='day_type', y='avg_rate_bps', color='asset_class', barmode='group',
    title='Weekend vs Weekday Avg Funding Rate by Asset Class',
    labels={'avg_rate_bps': 'Avg Rate (bps)', 'day_type': '', 'asset_class': 'Asset Class'},
    template='plotly_dark'
)
fig.update_layout(height=400)
fig.show()

In [ ]:
# Dual line chart: equity vs crypto funding rates over time
eq_daily = equity_df.groupby('date')['avg_rate_bps'].mean().reset_index()
eq_daily['asset_class'] = 'Equity'
cr_daily = crypto_df.groupby('date')['avg_rate_bps'].mean().reset_index()
cr_daily['asset_class'] = 'Crypto'
both = pd.concat([eq_daily, cr_daily])

fig = px.line(
    both, x='date', y='avg_rate_bps', color='asset_class',
    title='Equity vs Crypto Perpetual Funding Rates Over Time',
    labels={'avg_rate_bps': 'Avg Rate (bps)', 'date': 'Date', 'asset_class': 'Asset Class'},
    template='plotly_dark'
)
fig.update_layout(height=500, hovermode='x unified')
fig.show()

In [ ]:
# Key finding: weekend premium for equity vs crypto perps
eq_weekend = equity_df[equity_df['is_weekend']]['avg_rate_bps'].mean()
eq_weekday = equity_df[~equity_df['is_weekend']]['avg_rate_bps'].mean()
cr_weekend = crypto_df[crypto_df['is_weekend']]['avg_rate_bps'].mean()
cr_weekday = crypto_df[~crypto_df['is_weekend']]['avg_rate_bps'].mean()

eq_premium = ((eq_weekend / eq_weekday) - 1) * 100 if eq_weekday != 0 else 0
cr_premium = ((cr_weekend / cr_weekday) - 1) * 100 if cr_weekday != 0 else 0

print("=" * 60)
print("KEY FINDING: Equity vs Crypto Weekend Funding Premium")
print("=" * 60)
print(f"Equity weekend avg:  {eq_weekend:.2f} bps | weekday avg: {eq_weekday:.2f} bps | premium: {eq_premium:+.1f}%")
print(f"Crypto weekend avg:  {cr_weekend:.2f} bps | weekday avg: {cr_weekday:.2f} bps | premium: {cr_premium:+.1f}%")
print()
if abs(eq_premium) > abs(cr_premium):
    print(f"→ Equity perps show {abs(eq_premium - cr_premium):.1f}% {'higher' if eq_premium > cr_premium else 'lower'} weekend funding premium vs crypto perps.")
else:
    print(f"→ Crypto perps show {abs(cr_premium - eq_premium):.1f}% {'higher' if cr_premium > eq_premium else 'lower'} weekend funding premium vs equity perps.")